In [1]:
import pyspark, os, sys, shutil
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, LongType
os.environ['PYSPARK_PYTHON'] = os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
spark = SparkSession.builder.appName("FileSourcesDrill").getOrCreate()
sc = spark.sparkContext
sc.setLogLevel("ERROR")
print("Spark version:", spark.version)

# ── Prepare the drill input data ─────────────────────────────────────────
if os.path.exists("tmp/drill"):
    shutil.rmtree("tmp/drill")
os.makedirs("tmp/drill", exist_ok=True)

# Write a semicolon-delimited CSV with NO header row (mimics a legacy data feed)
NAMES = ["Alice","Bob","Carol","Dave","Eve","Frank","Grace","Hank","Ivy","Jack",
         "Kate","Liam","Maya","Noah","Olivia","Paul","Quinn","Rosa","Sam","Tina"]
DEPTS = ["Engineering","Marketing","HR","Finance","Operations"]

rows = [(i + 1, NAMES[i % 20], DEPTS[i % 5],
         50000 + (i * 1327 % 60000), 25 + (i * 7 % 30))
        for i in range(500)]

with open("tmp/drill/employees_raw.csv", "w") as f:
    for r in rows:
        f.write(";".join(str(v) for v in r) + "\n")

print(f"Written tmp/drill/employees_raw.csv  ({len(rows)} rows, no header, sep=';')")
print("First 3 lines:")
with open("tmp/drill/employees_raw.csv") as f:
    for i, line in enumerate(f):
        if i >= 3: break
        print(" ", line.strip())
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/28 18:33:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.1.2
Written tmp/drill/employees_raw.csv  (500 rows, no header, sep=';')
First 3 lines:
  1;Alice;Engineering;50000;25
  2;Bob;Marketing;51327;32
  3;Carol;HR;52654;39


# Drill: File Sources and Formats

You have a semicolon-delimited CSV file with no header row. The columns in order are: `id`, `name`, `dept`, `salary`, `age`.

**Tasks:**
1. Read the raw CSV correctly and save it as **Parquet with gzip compression**.
2. Read the Parquet file and demonstrate **column pruning** using `explain()`.
3. Write the data **partitioned by `dept`**, then read back only the `Engineering` partition.
4. Compare **file sizes** across text, CSV, JSON, and Parquet for the same dataset.

**Reading:** Apache Spark 4.1.2 — Data Sources (https://spark.apache.org/docs/latest/sql-data-sources.html)

# Task 1: Read and Clean the CSV → Save as Parquet

Read `tmp/drill/employees_raw.csv` correctly:
- Separator is `;`
- No header row — provide a schema or rename the `_c0`…`_c4` columns
- `salary` should be `IntegerType`, `age` should be `IntegerType`

Save the cleaned DataFrame as Parquet to `tmp/drill/employees_parquet` using **gzip** compression.

Expected: 500 rows, schema with columns `id(int)`, `name`, `dept`, `salary(int)`, `age(int)`.

In [2]:
# ── Stub ────────────────────────────────────────────────────────────────
schema = StructType([
    StructField("id",     IntegerType()),
    StructField("name",   StringType()),
    StructField("dept",   StringType()),
    StructField("salary", IntegerType()),
    StructField("age",    IntegerType()),
])

employees = None   # TODO: spark.read ... .csv("tmp/drill/employees_raw.csv")

# TODO: write employees as Parquet with gzip compression

if employees:
    employees.printSchema()
    employees.show(5)
    print(f"Row count: {employees.count()}")

# Task 2: Demonstrate Column Pruning and Predicate Pushdown

Read the Parquet file you created in Task 1, then:

1. Select only `name` and `salary` columns — run `.explain()` and find the `ReadSchema` line. It should show only `name` and `salary`, not the other three columns. This is **column pruning**.

2. Filter for `dept = "Engineering"` and `salary > 80000` — run `.explain()` and find the `PushedFilters` line. Both conditions should appear there. This is **predicate pushdown** — Spark pushes the filter into the Parquet reader so entire row groups that cannot match are skipped.

In [4]:
# ── Stub ────────────────────────────────────────────────────────────────
parquet_df = spark.read.parquet("tmp/drill/employees_parquet")

# TODO 1: select only name and salary, then call .explain()
#         Look for "ReadSchema: struct<name:string,salary:int>" in the output


# TODO 2: filter for dept="Engineering" AND salary > 80000, then .explain()
#         Look for "PushedFilters" in the output


# Task 3: Partitioned Write and Partition Pruning

Write the employee DataFrame **partitioned by `dept`** to `tmp/drill/employees_by_dept`.

1. Inspect the resulting directory structure — you should see one subdirectory per department (`dept=Engineering/`, `dept=Finance/`, etc.).

2. Read the partitioned Parquet back and filter for `dept = "HR"`. Run `.explain()` — look for `PartitionFilters` in the plan. Spark reads **only** the `dept=HR/` subdirectory.

3. Count employees per department using the partitioned read.

In [6]:
# ── Stub ────────────────────────────────────────────────────────────────

# TODO 1: write employees partitioned by dept to tmp/drill/employees_by_dept


# TODO 2: list the partition subdirectories


# TODO 3: read back, filter for dept="HR", explain(), count()

# Task 4: File Size Comparison

Write the `employees` DataFrame in four formats to `tmp/drill/compare_*/`, then compute and display the file sizes to confirm that Parquet is significantly smaller than the row-based formats.

Expected ordering (largest → smallest): JSON > Text > CSV > Parquet

In [18]:
# ── Stub ────────────────────────────────────────────────────────────────
from pyspark.sql.functions import concat_ws, col

def dir_size_bytes(path):
    total = 0
    for fname in os.listdir(path):
        if not fname.startswith("_") and not fname.startswith("."):
            total += os.path.getsize(os.path.join(path, fname))
    return total

# TODO: write employees in text, csv, json, and parquet formats
#       then print a size comparison table
#
# Hint for text (requires a single string column):
# text_df = employees.select(
#     concat_ws(",", col("id").cast("string"), col("name"),
#               col("dept"), col("salary").cast("string"),
#               col("age").cast("string")).alias("value")
# )

In [20]:
spark.stop()